Fixing the path issue

In [ ]:
import sys
from pathlib import Path


def find_project_root(marker="pyproject.toml") -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"{marker} not found starting from {p}")


root_dir = find_project_root()
sys.path.append(str(root_dir / "app" / "src"))
root_dir


Imports

In [ ]:
from pathlib import Path

from application.use_cases.boilerplate_removal import remove_boilerplate
from application.use_cases.chunking import chunk_typed_clauses
from application.use_cases.clause_classification import classify_and_enrich_clauses
from application.use_cases.clause_segmentation import segment_document
from domain.chunk import Chunk, ChunkingReport, ChunkRule
from domain.clause_classification import ClauseType
from infrastructure.config.settings import get_chunking_settings
from infrastructure.parsing.extraction import PyMuPdfTextExtractor
from infrastructure.parsing.manifest import read_manifest
from infrastructure.parsing.rules_loader import load_classification_rules

RAW_DIR = root_dir / "data" / "policies" / "raw"
MANIFEST_PATH = root_dir / "data" / "policies" / "manifest.csv"
RULES_PATH = root_dir / "data" / "parsing" / "clause_type_mapping.csv"

Let's pick a document to inspect

In [ ]:
manifest_records = read_manifest(MANIFEST_PATH)

document_id = "15"
manifest_row = next(row for row in manifest_records if row["id"] == document_id)
filename = manifest_row["filename"]
manifest_row

In [ ]:
class _RuleOnlyClassifier:
    """No-network stand-in for the LLM classification fallback pass."""

    def classify(self, clause_title: str, clause_text: str) -> tuple[ClauseType, float]:
        return ClauseType.OTHER, 0.0


document = PyMuPdfTextExtractor().extract(RAW_DIR / filename, document_id)
cleaned, boilerplate_counts = remove_boilerplate(document)
tree = segment_document(cleaned)

rules = load_classification_rules(RULES_PATH)
typed_clauses = classify_and_enrich_clauses(
    tree, manifest_records, rules, _RuleOnlyClassifier()
)

print(f"{filename}: {tree.report.clause_count} clauses, max_depth={tree.report.max_depth}")


Chunks

In [ ]:
settings = get_chunking_settings()

chunks, report = chunk_typed_clauses(
    typed_clauses,
    min_char_count=settings.chunk_min_char_count,
    target_char_count=settings.chunk_target_char_count,
    max_char_count=settings.chunk_max_char_count,
    sliding_window_overlap_chars=settings.chunk_sliding_window_overlap_chars,
)

report


In [ ]:
print(f"document_id: {report.document_id}")
print(f"chunk_count: {report.chunk_count}")
print(f"single: {report.single_count}")
print(f"merged: {report.merged_count}")
print(f"item_boundary_split: {report.item_boundary_split_count}")
print(f"sliding_window_split: {report.sliding_window_split_count}")
print(f"char_count min/p50/p90/max/mean:")
print(
    f"  {report.min_char_count} / {report.p50_char_count} / "
    f"{report.p90_char_count} / {report.max_char_count} / {report.mean_char_count:.1f}"
)


In [ ]:
from collections import Counter
import statistics

rule_counts = Counter(chunk.rule for chunk in chunks)
for rule in ChunkRule:
    print(f"{rule.value:>22}: {rule_counts.get(rule, 0)}")

lengths = sorted(chunk.char_count for chunk in chunks)
print()
print(f"n={len(lengths)}  min={lengths[0]}  max={lengths[-1]}")
print(f"mean={statistics.fmean(lengths):.1f}  median={statistics.median(lengths):.1f}")


In [ ]:
def show(chunk: Chunk, *, preview_chars: int = 400) -> None:
    print(f"chunk_id: {chunk.chunk_id}")
    print(f"clause_id: {chunk.clause_id}")
    print(f"source_clause_ids: {chunk.source_clause_ids}")
    print(f"chunk_index/count: {chunk.chunk_index}/{chunk.chunk_count}")
    print(f"rule: {chunk.rule.value}")
    print(f"clause_type: {chunk.clause_type.value}")
    print(f"bundle_section: {chunk.bundle_section}")
    print(f"char_count: {chunk.char_count}")
    print(f"parent_path: {chunk.parent_path!r}")
    print("\ntext preview\n")
    print(chunk.text[:preview_chars])
    print()


for rule in ChunkRule:
    example = next((chunk for chunk in chunks if chunk.rule == rule), None)
    print(f"--{rule.value}--")
    if example is None:
        print("(no chunk in this document uses this rule)\n")
        continue
    show(example)


Sanity check

In [ ]:
for chunk in chunks:
    assert chunk.clause_id
    assert chunk.clause_type is not None
    assert chunk.provenance is not None


by_clause: dict[str, list[Chunk]] = {}
for chunk in chunks:
    by_clause.setdefault(chunk.clause_id, []).append(chunk)

for clause_id, pieces in by_clause.items():
    pieces_sorted = sorted(pieces, key=lambda c: c.chunk_index)
    assert [c.chunk_index for c in pieces_sorted] == list(range(len(pieces_sorted)))
    assert all(c.chunk_count == len(pieces_sorted) for c in pieces_sorted)

known_clause_ids = {typed.clause.clause_id for typed in typed_clauses}
for chunk in chunks:
    assert set(chunk.source_clause_ids) <= known_clause_ids

print("all sanity checks passed")


Working with flatten chunk

In [ ]:
from infrastructure.rag.chunk_schema import flatten_chunk

record = flatten_chunk(chunks[0])
record


In [ ]:
import json

SNAPSHOT_DIR = root_dir / "tests" / "unit" / "infrastructure" / "rag" / "snapshots"


def serialize(chunk: Chunk) -> dict:
    return {
        "chunk_id": chunk.chunk_id,
        "clause_id": chunk.clause_id,
        "source_clause_ids": list(chunk.source_clause_ids),
        "chunk_index": chunk.chunk_index,
        "chunk_count": chunk.chunk_count,
        "rule": chunk.rule.value,
        "char_count": chunk.char_count,
        "parent_path": chunk.parent_path,
        "clause_type": chunk.clause_type.value,
        "bundle_section": chunk.bundle_section,
        "text_preview": chunk.text[:120],
    }


snapshot_path = SNAPSHOT_DIR / f"chunks_{document_id}.jsonl"
with snapshot_path.open(encoding="utf-8") as handle:
    expected = [json.loads(line) for line in handle if line.strip()]

actual = [serialize(chunk) for chunk in chunks]

if actual == expected:
    print(f"matches committed snapshot ({len(actual)} chunks)")
else:
    for i, (a, e) in enumerate(zip(actual, expected)):
        if a != e:
            print(f"first mismatch at index {i}:")
            print("actual:", a)
            print("expected:", e)
            break
    else:
        print(f"chunk counts differ: actual={len(actual)} expected={len(expected)}")


Trying different chunking parameters

In [ ]:
tight_chunks, tight_report = chunk_typed_clauses(
    typed_clauses,
    min_char_count=150,
    target_char_count=400,
    max_char_count=600,
    sliding_window_overlap_chars=80,
)

print(f"default max_char_count={settings.chunk_max_char_count} -> {report.chunk_count} chunks")
print(f"tight max_char_count=600 -> {tight_report.chunk_count} chunks")
tight_report


Using LLM classifier

In [ ]:
from infrastructure.config.llm_client_factory import build_chat_model
from infrastructure.config.settings import get_llm_settings
from infrastructure.parsing.llm_classification_cache import CachingClauseClassifier
from infrastructure.parsing.llm_classifier import LangchainClauseClassifier

llm_settings = get_llm_settings()
llm = build_chat_model(
    llm_settings,
    llm_settings.llm_model_fast,
    provider_order=llm_settings.llm_classification_provider_order,
    allow_fallbacks=llm_settings.llm_classification_allow_fallbacks,
)
real_classifier = CachingClauseClassifier(
    LangchainClauseClassifier(llm),
    model=llm_settings.llm_model_fast,
    cache_path=root_dir / "data" / "cache" / "llm_classification" / "cache.jsonl",
)

real_typed_clauses = classify_and_enrich_clauses(
    tree,
    manifest_records,
    rules,
    real_classifier,
    max_workers=llm_settings.llm_classification_max_workers,
)

real_chunks, real_report = chunk_typed_clauses(
    real_typed_clauses,
    min_char_count=settings.chunk_min_char_count,
    target_char_count=settings.chunk_target_char_count,
    max_char_count=settings.chunk_max_char_count,
    sliding_window_overlap_chars=settings.chunk_sliding_window_overlap_chars,
)
real_report


In [ ]:
from collections import Counter

from infrastructure.rag.chunk_artifact import CHUNKS_JSONL_PATH, read_chunks_jsonl

all_records = read_chunks_jsonl(root_dir / CHUNKS_JSONL_PATH)
print(f"{len(all_records)} chunks across {len({r.document_id for r in all_records})} documents")

by_rule = Counter(r.rule for r in all_records)
for rule, count in by_rule.items():
    print(f"{rule.value:>22}: {count}")

by_product_line = Counter(r.product_line for r in all_records)
by_product_line.most_common()
